### Data loading

In [31]:
import pandas as pd

In [32]:
chronos = pd.read_csv("../data/CRISPR_gene_effect.csv",index_col=0)
chronos.columns = [c.split()[0].upper() + '_Chronos' for c in chronos.columns]
expression = pd.read_csv("../data/CCLE_expression.csv", index_col=0)
expression.columns = [c.split()[0].upper()+'_Expression' for c in expression.columns]
sample_info = pd.read_csv("../data/sample_info.csv", index_col=0)
ch_ex = chronos.merge(expression, right_index=True, left_index=True).dropna()

In [33]:
mutation= pd.read_csv("../data/Damaging_mutation.csv", index_col=0)
mutation.columns = [c.split()[0].upper() + '_Mutation' for c in mutation.columns]
ch_ex_mutation = ch_ex.merge(mutation, left_index=True, right_index=True, how='left').copy()

In [34]:
tp53_state = pd.read_csv('../embeddings/Trimap_tp53.csv',index_col=0)
hippo_state = pd.read_csv('../embeddings/Trimap_hippo.csv',index_col=0)

In [35]:
import pandas as pd
D = pd.read_csv("../data/D_22Q2.csv", index_col=0)
ch_ex_D = ch_ex_mutation.merge(D,left_index=True, right_index=True, how='left').copy()

### Predicting Tp53 mutational status (Fig. S6b)

In [21]:
import gseapy as gp

msig = gp.Msigdb()
hallmarks = msig.get_gmt(category="h.all")
p53_genes = hallmarks["HALLMARK_P53_PATHWAY"]
p53_set = {"HALLMARK_P53_PATHWAY": p53_genes}
print(len(p53_genes))

200


In [12]:
e2 = ch_ex[[a for a in ch_ex.columns if a.split('_')[-1] == 'Expression']]
e2.columns = [a.split('_')[0] for a in e2.columns]
expr = e2.T

In [ ]:
gsva_result = gp.gsva(data=expr,gene_sets=p53_set,outdir=None)
p53_gsva = gsva_result.res2d.query("Term == 'HALLMARK_P53_PATHWAY'").set_index("Name")["ES"]
p53_gsva.head()

Name
ACH-002926   -0.037696
ACH-000001   -0.269760
ACH-000004   -0.339724
ACH-000005   -0.322979
ACH-000007    0.311667
Name: ES, dtype: float64

In [22]:
ssgsea_result = gp.ssgsea(data=expr,gene_sets=p53_set,outdir=None,sample_norm_method="rank",no_plot=True)
ssgsea_long = ssgsea_result.res2d
p53_ssgsea = ssgsea_long.query("Term == 'HALLMARK_P53_PATHWAY'").set_index("Name")["NES"]
p53_ssgsea.head()

Name
ACH-001388    1.526042
ACH-001422    1.460878
ACH-000794    1.434116
ACH-000771    1.421543
ACH-001509    1.378970
Name: NES, dtype: float64

In [ ]:
import numpy as np
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score

Xs = {
    "StateMap": tp53_state.values,
    "Webster": ch_ex_D.V6.values.reshape(-1, 1),
    "GSVA": p53_gsva.sort_index().values.reshape(-1, 1),
    "ssGSEA": p53_ssgsea.sort_index().values.reshape(-1, 1),
}

y = np.array([int(a > 0) for a in ch_ex_D["TP53_Mutation"].values])
n_repeats = 100

results = {name: [] for name in Xs}
for seed in range(n_repeats):
    cv = StratifiedKFold(n_splits=5,shuffle=True,random_state=seed)
    splits = list(cv.split(np.zeros(len(y)), y))
    for name, X in Xs.items():
        model = LogisticRegression(max_iter=1000)
        pred = cross_val_predict(model,X,y,cv=splits,method="predict_proba")[:, 1]
        auc = roc_auc_score(y, pred)
        results[name].append(auc)

for name, aucs in results.items():
    aucs = np.array(aucs)
    print(f"{name}: ", f"AUROC = {aucs.mean():.3f} ± {aucs.std(ddof=1):.3f}")

StateMap: AUROC = 0.869 ± 0.001
Webster: AUROC = 0.836 ± 0.001
GSVA: AUROC = 0.594 ± 0.002
ssGSEA: AUROC = 0.487 ± 0.016


### Predicting Hippo-related features (Fig. S3c, d)

In [25]:
import gseapy as gp

msig = gp.Msigdb()
hallmarks = msig.get_gmt(category="c6.all")

yap_genes = hallmarks["YAP1_UP"]

yap_set = {"YAP_UP": yap_genes}
print(len(yap_genes))

47


In [26]:
expr = e2.T

ssgsea_result = gp.ssgsea(data=expr,gene_sets=yap_set,outdir=None,sample_norm_method="rank",no_plot=True)
ssgsea_long = ssgsea_result.res2d
yap_ssgsea = ssgsea_long.query("Term == 'YAP_UP'").set_index("Name")["NES"]
gsva_result = gp.gsva(data=expr,gene_sets=yap_set,outdir=None)
yap_gsva = gsva_result.res2d

In [36]:
import numpy as np
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error
from scipy.stats import spearmanr, pearsonr

Xs = {
    "StateMap": hippo_state.values,
    "Webster": ch_ex_D["V8"].values.reshape(-1, 1),
    "GSVA": yap_gsva.sort_values("Name")["ES"].values.reshape(-1, 1),
    "ssGSEA": yap_ssgsea.sort_index().values.reshape(-1, 1),
}

y = ch_ex_D["CCN2_Expression"].values
n_repeats = 100
results = {name: {"R2": [], "RMSE": [], "Spearman": [], "Pearson": []} for name in Xs}

for seed in range(n_repeats):
    cv = KFold(n_splits=5, shuffle=True, random_state=seed)
    splits = list(cv.split(np.zeros(len(y))))
    for name, X in Xs.items():
        pred = cross_val_predict(LinearRegression(), X, y, cv=splits)
        results[name]["R2"].append(r2_score(y, pred))
        results[name]["RMSE"].append(np.sqrt(mean_squared_error(y, pred)))
        results[name]["Spearman"].append(spearmanr(y, pred).statistic)
        results[name]["Pearson"].append(pearsonr(y, pred).statistic)

for name in Xs:
    values = np.array(results[name]["R2"])
    print(f"{name}: R2 = {values.mean():.3f} ± {values.std(ddof=1):.3f}")

StateMap: R2 = 0.587 ± 0.001
Webster: R2 = 0.167 ± 0.002
GSVA: R2 = 0.325 ± 0.001
ssGSEA: R2 = 0.342 ± 0.001
